In [1]:
import torch
import torch.nn.functional as F
from unsloth import FastLanguageModel
from transformers import AutoModelForCausalLM, AutoTokenizer

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


### Dataset Generation

In [2]:
from datasets import load_dataset

TRAIN_SIZE = 2000
dataset = load_dataset("allenai/tulu-3-sft-mixture", split="train", streaming=True)
dataset.shuffle(seed=42)
train_dataset = []
for i, x in enumerate(dataset):
    if i >= TRAIN_SIZE:
        break
    train_dataset.append(dict(messages=x['messages']))

train_dataset[0]

{'messages': [{'content': 'Create a snippet of Terraform HCL code that create an AWS autoscaling group, and an ALB in front to expose an application to internet.',
   'role': 'user'},
  {'content': 'Sure, here\'s an example Terraform HCL code that creates an AWS Autoscaling Group and an Application Load Balancer to expose an application to the internet:\n``` \n# Configure the AWS provider\nprovider "aws" {\n  region = "us-east-1"\n}\n\n# Create a security group to allow traffic to the ALB\nresource "aws_security_group" "alb_sg" {\n  name_prefix = "alb_sg"\n  ingress {\n    from_port = 80\n    to_port = 80\n    protocol = "tcp"\n    cidr_blocks = ["0.0.0.0/0"]\n  }\n}\n\n# Create an ALB and target group\nresource "aws_lb" "alb" {\n  name               = "example-alb"\n  internal           = false\n  load_balancer_type = "application"\n\n  subnets = ["subnet-12345678", "subnet-87654321"]\n\n  security_groups = [aws_security_group.alb_sg.id]\n\n  tags = {\n    Environment = "production"\n

In [3]:
import os

max_seq_length = 1024
STUDENT_MODEL_NAME = "./knowledge-ingestion-test/model/checkpoint-178"
os.path.exists(STUDENT_MODEL_NAME)


True

In [4]:
from peft import PeftConfig, PeftModel

config = PeftConfig.from_pretrained(STUDENT_MODEL_NAME)
config.base_model_name_or_path

'unsloth/Qwen3-4B-Instruct-2507'

In [5]:
base_model = AutoModelForCausalLM.from_pretrained(
    config.base_model_name_or_path,
    torch_dtype=torch.bfloat16,
    device_map="cuda:0",
)
base_model.eval()

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560, padding_idx=151654)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2560,), eps=1e-06)
        (

In [6]:
# student_model = base_model
student_model = PeftModel.from_pretrained(
    base_model,
    STUDENT_MODEL_NAME,
    is_trainable=False,   # True if you want to keep training
)
student_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 2560, padding_idx=151654)
        (layers): ModuleList(
          (0-35): 36 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2560, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2560, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_p

In [7]:
for name, param in student_model.named_parameters():
    if "lora" in name:
        param.requires_grad = True
        assert param.requires_grad == True
    else:
        assert param.requires_grad == False



In [8]:
device = next(student_model.parameters()).device
device

device(type='cuda', index=0)

In [9]:
# ----------------------------
# 2) Load frozen teacher
# ----------------------------
# For THIS minimal implementation, assume teacher/student share tokenizer ids
teacher = AutoModelForCausalLM.from_pretrained(
    config.base_model_name_or_path,
    torch_dtype=torch.bfloat16,
    device_map="cuda:1",
)
teacher.eval()

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560, padding_idx=151654)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2560,), eps=1e-06)
        (

In [10]:
tokenizer = AutoTokenizer.from_pretrained(config.base_model_name_or_path)

In [11]:
optimizer = torch.optim.AdamW(student_model.parameters(), lr=1e-5)

In [12]:
@torch.no_grad()
def sample_from_student(messages, max_new_tokens=256, temperature=1.0, top_p=1.0):
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt", padding=True, padding_side="left").to(device)
    prompt_len = inputs["input_ids"].shape[1]

    sampled = student_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    return sampled, prompt_len

messages = [
    [{'role': 'user', 'content': 'Hello, how are you?'}],
    [{'role': 'user', 'content': 'yo whats the weather in tokyo?'}],
]

sampled, prompt_len = sample_from_student(messages)
print(tokenizer.decode(sampled))

['<|vision_pad|><|vision_pad|><|vision_pad|><|im_start|>user\nHello, how are you?<|im_end|>\n<|im_start|>assistant\n鹦鹉不会说话，但能模仿人说话。例如，如果你问它"你好吗？"，它可能会重复说"你好吗？"<|im_end|>', '<|im_start|>user\nyo whats the weather in tokyo?<|im_end|>\n<|im_start|>assistant\n<think>\n\n<tool_call>\n\nThe current temperature in Tokyo is 74 degrees Fahrenheit.<|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|>']


In [84]:
tokenizer.pad_token, tokenizer.eos_token

('<|vision_pad|>', '<|im_end|>')

In [13]:
optimizer.zero_grad()
student_model.zero_grad()

In [14]:

max_grad_norm = 1.0

In [85]:
full_ids = sampled
full_ids = full_ids.to(device)
attn = (full_ids != tokenizer.pad_token_id) & (full_ids != tokenizer.eos_token_id)
full_ids.shape, attn.shape



(torch.Size([2, 38]), torch.Size([2, 38]))

In [86]:
# get student logprobs
out = student_model(input_ids=full_ids, attention_mask=attn)
student_logprobs = F.log_softmax(out.logits, dim=-1)
print(student_logprobs.shape)

torch.Size([2, 38, 151936])


In [87]:
# get teacher logprobs
with torch.no_grad():
    teacher_out = teacher(input_ids=full_ids.to(teacher.device), attention_mask=attn.to(teacher.device))
    teacher_logprobs = F.log_softmax(teacher_out.logits, dim=-1)
print(teacher_logprobs.shape)



torch.Size([2, 38, 151936])


In [88]:
full_ids.shape

torch.Size([2, 38])

In [90]:
# select the logprob for the tokens mentioned in full_ids
student_lp = student_logprobs.gather(dim=-1, index=full_ids.unsqueeze(-1)).squeeze(-1)
student_lp.shape

torch.Size([2, 38])

In [91]:
full_ids[0, 0]

tensor(151654, device='cuda:0')

In [92]:
teacher_logprobs[0, 0, 151644]

tensor(-20., device='cuda:1', dtype=torch.bfloat16)

In [93]:
teacher_lp = teacher_logprobs.to(student_logprobs.device).gather(dim=-1, index=full_ids.unsqueeze(-1)).squeeze(-1)
teacher_lp.shape, teacher_lp[0, 0]

(torch.Size([2, 38]), tensor(-20.1250, device='cuda:0', dtype=torch.bfloat16))

In [94]:
# get advantage
with torch.no_grad():
    advantage = (teacher_lp.to(student_lp.device) - student_lp.detach()).clamp(-5, 5)
advantage.shape

torch.Size([2, 38])

In [95]:
# # get loss
# full_loss = advantage * student_lp


In [ ]:
# print(-(ignore_mask * student_lp * advantage).sum() / ignore_mask.sum())

# # importance sampling loss
# print((ignore_mask * torch.exp(student_lp - teacher_lp) * advantage).sum() / ignore_mask.sum())

tensor(33.5000, device='cuda:0', dtype=torch.bfloat16, grad_fn=<DivBackward0>)
tensor(-54016., device='cuda:0', dtype=torch.bfloat16, grad_fn=<DivBackward0>)


In [ ]:
# ignore_mask.shape, student_lp.shape, teacher_lp.shape, advantage.shape

(torch.Size([2, 38]),
 torch.Size([2, 38]),
 torch.Size([2, 38]),
 torch.Size([2, 38]))

In [103]:
advantage

tensor([[-4.1875, -4.1875, -4.1875,  5.0000, -5.0000, -0.8750, -2.0000, -4.6875,
         -0.0625, -3.9375, -3.3125, -5.0000, -5.0000,  5.0000, -5.0000, -0.2500,
         -5.0000, -5.0000, -3.7500, -1.2500, -4.3750, -2.7500, -2.2500, -2.9375,
         -2.3125, -3.3750, -4.5000, -4.3125, -2.3125, -3.5625, -5.0000, -5.0000,
         -5.0000, -5.0000, -5.0000, -5.0000, -5.0000, -5.0000],
        [ 5.0000, -5.0000, -0.7500,  0.3125,  0.0000, -0.2188, -4.3125,  0.1250,
         -3.6875, -5.0000, -3.3125, -2.6875, -5.0000,  5.0000, -5.0000,  4.8125,
         -5.0000,  0.1250,  2.5000,  0.5000, -1.1250, -5.0000, -5.0000, -5.0000,
         -5.0000, -4.1250, -5.0000, -5.0000, -4.7500, -2.9375, -3.7188, -1.1250,
         -2.8750, -2.4688, -4.8125, -5.0000,  2.1250, -5.0000]],
       device='cuda:0', dtype=torch.bfloat16)

In [ ]:




# get loss
loss = -(full_loss * ignore_mask).sum() / ignore_mask.sum().clamp(min=1)
loss

tensor(-27.1250, device='cuda:0', dtype=torch.bfloat16, grad_fn=<DivBackward0>)

In [78]:
from torch.nn import functional as F

F.kl_div(student_lp, teacher_lp, reduction='batchmean', log_target=True)

tensor(-2.7418e-05, device='cuda:0', dtype=torch.bfloat16,
       grad_fn=<DivBackward0>)

In [80]:
F.kl_div(student_logprobs, teacher_logprobs.to(student_logprobs.device), reduction='batchmean', log_target=True)

tensor(118.5000, device='cuda:0', dtype=torch.bfloat16, grad_fn=<DivBackward0>)

In [81]:
tokenizer.decode(full_ids[0])

'<|im_start|>user\nHello, how are you?<|im_end|>\n<|im_start|>assistant\n<think>\n\n_Pods\n\nGood. How is your day going?<|im_end|>'

In [13]:
# One Step
def opd_step(messages):
    # generate rollout
    sampled, prompt_len = sample_from_student(messages)

    optimizer.zero_grad()
    student_model.zero_grad()

    full_ids = sampled
    full_ids = full_ids.to(device)
    target_ids = full_ids[:, 1:]
    attn = (full_ids != tokenizer.pad_token_id)

    # get student logprobs
    out = student_model(input_ids=full_ids, attention_mask=attn)
    student_logprobs = F.log_softmax(out.logits[:, :-1, :], dim=-1)
    student_lp = student_logprobs.gather(dim=-1, index=target_ids.unsqueeze(-1)).squeeze(-1)

    # get teacher logprobs
    with torch.no_grad():
        teacher_out = teacher(input_ids=full_ids.to(teacher.device), attention_mask=attn.to(teacher.device))
        teacher_logprobs = F.log_softmax(teacher_out.logits[:, :-1, :], dim=-1)
    teacher_lp = teacher_logprobs.to(student_logprobs.device).gather(dim=-1, index=target_ids.unsqueeze(-1)).squeeze(-1)

    # get advantage
    with torch.no_grad():
        advantage = (teacher_lp.to(student_lp.device) - student_lp.detach()).clamp(-5, 5)

    # mask
    mask = torch.ones_like(target_ids) & (target_ids != tokenizer.eos_token_id) & (target_ids != tokenizer.pad_token_id)
    mask[:, :prompt_len-1] = 0

    # get loss
    loss = -(mask * student_lp * advantage).sum() / mask.sum().clamp(min=1)

    # backprop
    loss.backward()
    total_norm = torch.nn.utils.clip_grad_norm_(student_model.parameters(), 1.0)
    optimizer.step()

    # stats
    mean_advantage = ((mask*advantage).sum() / mask.sum().clamp(min=1)).item()
    mean_reverse_kl = -mean_advantage
    return {
        'loss': loss.item(),
        'total_norm': total_norm.item(),
        'mean_advantage': mean_advantage,
        'mean_reverse_kl': mean_reverse_kl,
    }

In [ ]:
BATCH_SIZE = 2

import json
from tqdm import tqdm

END = len(train_dataset)
# END = 10
pbar = tqdm(range(0, END, BATCH_SIZE))
for idx in pbar:
    batch = [x['messages'][:-1] for x in train_dataset[idx:idx+BATCH_SIZE]]
    stats = opd_step(batch)
    pbar.set_postfix(stats)


  2%| | 10/500 [01:19<1:05:14,  7.99s/it, loss=-1.83, total_norm=11.4, mean_advantage=-0.672, mean_rev


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.54 GiB. GPU 0 has a total capacity of 79.10 GiB of which 150.75 MiB is free. Process 3110444 has 8.71 GiB memory in use. Process 3203772 has 11.65 GiB memory in use. Including non-PyTorch memory, this process has 58.57 GiB memory in use. Of the allocated memory 57.29 GiB is allocated by PyTorch, and 571.60 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)